In [14]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine("mysql+pymysql://root:@localhost/climate_watch")

df_test = pd.read_sql("SELECT * FROM GlobalLandTemperaturesByCity LIMIT 5", engine)
df_test


tables = [
    "GlobalLandTemperaturesByCity",
    "GlobalLandTemperaturesByCountry",
    "GlobalLandTemperaturesByMajorCity",
    "GlobalLandTemperaturesByState",
    "globaltemperatures"
]

with engine.connect() as conn:
    for table in tables:
        subset_table = f"{table}_subset"
        
        conn.execute(text(f"DROP TABLE IF EXISTS {subset_table}"))
        conn.execute(text(f"""
            CREATE TABLE {subset_table} AS
            SELECT *
            FROM {table}
            WHERE YEAR(dt) >= 1980;
        """))
        
        print(f"Klaar: {subset_table}")

conn.commit()

Klaar: GlobalLandTemperaturesByCity_subset
Klaar: GlobalLandTemperaturesByCountry_subset
Klaar: GlobalLandTemperaturesByMajorCity_subset
Klaar: GlobalLandTemperaturesByState_subset
Klaar: globaltemperatures_subset


In [15]:
for table in tables:
    subset_table = f"{table}_subset"
    
    df = pd.read_sql(f"SELECT * FROM {subset_table}", engine)
    
    print(f"\n--- {subset_table} ---")
    
    print("Missing per column:")
    print(df.isnull().sum())
    
    missing_rows = df[df.isnull().any(axis=1)]
    
    print(f"Rows with missing values: {len(missing_rows)}")


--- GlobalLandTemperaturesByCity_subset ---
Missing per column:
dt                               0
AverageTemperature               0
AverageTemperatureUncertainty    0
City                             0
Country                          0
Latitude                         0
Longitude                        0
dtype: int64
Rows with missing values: 0

--- GlobalLandTemperaturesByCountry_subset ---
Missing per column:
dt                               0
AverageTemperature               0
AverageTemperatureUncertainty    0
Country                          0
dtype: int64
Rows with missing values: 0

--- GlobalLandTemperaturesByMajorCity_subset ---
Missing per column:
dt                               0
AverageTemperature               0
AverageTemperatureUncertainty    0
City                             0
Country                          0
Latitude                         0
Longitude                        0
dtype: int64
Rows with missing values: 0

--- GlobalLandTemperaturesByState_subset --

In [5]:
for table in tables:
    subset_table = f"{table}_subset"
    
    df = pd.read_sql(f"SELECT * FROM {subset_table}", engine)
    
    print(f"\n--- {subset_table} ---")
    
    print("Missing per column:")
    print(df.isnull().sum())
    
    missing_rows = df[df.isnull().any(axis=1)]
    
    print(f"Rows with missing values: {len(missing_rows)}")


--- GlobalLandTemperaturesByCity_subset ---
Missing per column:
dt                               0
AverageTemperature               0
AverageTemperatureUncertainty    0
City                             0
Country                          0
Latitude                         0
Longitude                        0
dtype: int64
Rows with missing values: 0

--- GlobalLandTemperaturesByCountry_subset ---
Missing per column:
dt                               0
AverageTemperature               0
AverageTemperatureUncertainty    0
Country                          0
dtype: int64
Rows with missing values: 0

--- GlobalLandTemperaturesByMajorCity_subset ---
Missing per column:
dt                               0
AverageTemperature               0
AverageTemperatureUncertainty    0
City                             0
Country                          0
Latitude                         0
Longitude                        0
dtype: int64
Rows with missing values: 0

--- GlobalLandTemperaturesByState_subset --

In [ ]:
for table in tables:
    subset_table = f"{table}_subset"
    
    df = pd.read_sql(f"SELECT * FROM {subset_table}", engine)
    
    print(f"\n=== {subset_table} ===")
    
    # 🔹 NULL values
    null_rows = df[df.isnull().any(axis=1)]
    print(f"\nRows with NULL values: {len(null_rows)}")
    print(null_rows.head(5))
    
    # 🔹 ZERO values
    zero_rows = df[(df == 0).any(axis=1)]
    print(f"\nRows with ZERO values: {len(zero_rows)}")
    print(zero_rows.head(5))
    
    # 🔹 Counts per column (nice overview)
    print("\nNULL count per column:")
    print(df.isnull().sum())
    
    print("\nZERO count per column:")
    zero_counts = (df == 0).sum()
    zero_counts = zero_counts[zero_counts > 0]
    print(zero_counts if not zero_counts.empty else "No zeros 🎉")

    import numpy as np




=== GlobalLandTemperaturesByCity_subset ===

Rows with NULL values: 0
Empty DataFrame
Columns: [dt, AverageTemperature, AverageTemperatureUncertainty, City, Country, Latitude, Longitude]
Index: []

Rows with ZERO values: 3080
              dt  AverageTemperature  AverageTemperatureUncertainty     City  \
404   2013-09-01                 0.0                            0.0    Århus   
809   2013-09-01                 0.0                            0.0    Çorlu   
1214  2013-09-01                 0.0                            0.0    Çorum   
1619  2013-09-01                 0.0                            0.0  Öskemen   
2024  2013-09-01                 0.0                            0.0   Ürümqi   

         Country Latitude Longitude  
404      Denmark   57.05N  10.33E\r  
809       Turkey   40.99N  27.69E\r  
1214      Turkey   40.99N  34.08E\r  
1619  Kazakhstan   50.63N  82.39E\r  
2024       China   44.20N  87.20E\r  

NULL count per column:
dt                               0
Avera

KeyError: 'AverageTemperature'

In [20]:
df["AverageTemperature"].dtype

KeyError: 'AverageTemperature'

print(df.columns)


In [21]:
print(df.columns)

Index(['dt', 'LandAverageTemperature', 'LandAverageTemperatureUncertainty',
       'LandMaxTemperature', 'LandMaxTemperatureUncertainty',
       'LandMinTemperature', 'LandMinTemperatureUncertainty',
       'LandAndOceanAverageTemperature',
       'LandAndOceanAverageTemperatureUncertainty'],
      dtype='str')


In [22]:
import numpy as np

# Step 1: replace 0 with NaN (if needed)
df["LandAverageTemperature"] = df["LandAverageTemperature"].replace(0, np.nan)

# Step 2: fill missing values with mean
df["LandAverageTemperature"] = df["LandAverageTemperature"].fillna(
    df["LandAverageTemperature"].mean()
)


TypeError: Cannot perform reduction 'mean' with string dtype

In [23]:
df["LandAverageTemperature"] = pd.to_numeric(
    df["LandAverageTemperature"], errors="coerce"
)


In [24]:
import numpy as np

df["LandAverageTemperature"] = df["LandAverageTemperature"].replace(0, np.nan)

In [25]:
df["LandAverageTemperature"] = df["LandAverageTemperature"].fillna(
    df["LandAverageTemperature"].mean()
)